In [0]:
 %run ./01.config


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import logging


logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


class DemandFeatureBuilder:
    
    def __init__(self, spark, source_table):
        self.spark = spark
        self.source_table = source_table
    
    def build_and_save(self, target_table):
        try:
            logger.info(f"Loading data from {self.source_table}")
            df = self.spark.table(self.source_table)
            
            window = Window.partitionBy("product_name", "destination_city").orderBy("transaction_date")
            rolling_window = window.rowsBetween(-7, -1)
            
            features = df.withColumns({
                "demand_lag_1": F.lag("total_demand", 1).over(window),
                "demand_lag_7": F.lag("total_demand", 7).over(window),
                "rolling_avg_7": F.avg("total_demand").over(rolling_window),
                "day_of_week": F.dayofweek("transaction_date"),
                "month": F.month("transaction_date")
            })
            
            features = features.select(
                "transaction_date", "product_name", "destination_city",
                "total_demand", "avg_unit_price", "total_inventory",
                "day_of_week", "month",
                "demand_lag_1", "demand_lag_7",
                "rolling_avg_7"
            )
            
            
            features = features.filter(
                F.col("demand_lag_7").isNotNull() &
                F.col("rolling_avg_7").isNotNull()
            )
            
    
            logger.info(f"Saving features to {target_table}")
            features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
            
            result = self.spark.table(target_table)
            stats = result.agg(
                F.count("*").alias("rows"),
                F.min("transaction_date").alias("start_date"),
                F.max("transaction_date").alias("end_date")
            ).collect()[0]
            
            logger.info(f"Done! Created {target_table} with {stats['rows']:,} rows ({stats['start_date']} to {stats['end_date']})")
            return result
            
        except Exception as e:
            logger.error(f"Error: {str(e)}")
            raise


builder = DemandFeatureBuilder(spark, GOLD_AGG)
demand_features = builder.build_and_save(GOLD_FEATURES)